# Pokémon TCG Stock Network Analysis — MADT7202

**Business question:**  
การ์ดใบใดควรอยู่ในรายการตรวจสต็อกประจำสัปดาห์ และคู่การ์ดใดควรตรวจยอดคงเหลือพร้อมกันก่อนออกคำสั่งซื้อ?

Notebook นี้ใช้ **frozen sanitized snapshot** ใน repository เพื่อให้ผลตรงกับรายงานที่ส่ง

**Expected submitted dataset**
- 3 Pumpkaweekly tournaments: 14 / 21 / 28 Sep 2026
- 584 valid 60-card decklists
- 14,744 card-line records
- Player identifiers are not stored in the published snapshot

> เปิดจาก GitHub บน Google Colab แล้วกด **Runtime → Run all** ได้เลย


In [ ]:
# 0) Colab / repository bootstrap
from pathlib import Path
import importlib.util
import subprocess
import sys
import shutil

REPO_URL = "https://github.com/KankaweeK356/MADT7202-Pokemon-Network.git"
REPO_NAME = "MADT7202-Pokemon-Network"

def find_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / REPO_NAME,
        Path("/content") / REPO_NAME,
        Path("/mnt/data") / REPO_NAME,
    ]
    for p in candidates:
        if (p / "data/raw/decklists_snapshot.csv").exists():
            return p
    return None

ROOT = find_repo_root()

# If notebook is opened directly from GitHub in Google Colab,
# Colab does not automatically download the repository's data folders.
# Clone the public repo once so all relative paths become reproducible.
if ROOT is None:
    clone_target = Path("/content") / REPO_NAME
    if clone_target.exists():
        shutil.rmtree(clone_target)
    print("Repository data not found locally. Cloning public repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(clone_target)],
        check=True
    )
    ROOT = clone_target

DATA_PATH = ROOT / "data/raw/decklists_snapshot.csv"
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

required = {
    "pandas": "pandas>=2.0",
    "numpy": "numpy>=1.24",
    "networkx": "networkx>=3.0",
    "matplotlib": "matplotlib>=3.7",
}
missing = [spec for mod, spec in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Repository root:", ROOT)
print("Frozen snapshot:", DATA_PATH)


In [ ]:
# 1) Imports and declared thresholds
import math
import re
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

MIN_CARD_PREVALENCE = 0.10
MIN_SHARED_DECKS = 30
JACCARD_CUTOFF = 0.30
MIN_LIFT = 1.20
MIN_SHARED_ARCHETYPES = 3
ROBUSTNESS_JACCARD = [0.25, 0.30, 0.35]

# Declared before ranking: obvious evolution / mechanical dependencies.
MECHANICAL_PAIRS = {
    tuple(sorted(x)) for x in [
        ("Dreepy", "Drakloak"),
        ("Dreepy", "Dragapult ex"),
        ("Drakloak", "Dragapult ex"),
        ("Dunsparce", "Dudunsparce"),
        ("Duskull", "Dusclops"),
        ("Duskull", "Dusknoir"),
        ("Dusclops", "Dusknoir"),
        ("Abra", "Kadabra"),
        ("Abra", "Alakazam"),
        ("Kadabra", "Alakazam"),
        ("Buneary", "Mega Lopunny ex"),
        ("N's Zorua", "N's Zoroark ex"),
        ("Applin", "Dipplin"),
        ("Riolu", "Mega Lucario ex"),
    ]
}

TOP_WATCHLIST_NODES_FOR_DISPLAY = 5
TOP_PAIR_EDGES_FOR_DISPLAY = 5
MAX_DISPLAY_EDGES_PER_NODE = 2

print("Thresholds locked.")


## Why these thresholds?

- **Prevalence ≥10% (≥59 decks):** Core stock should have a broad competitive footprint rather than be a niche tech card.
- **Shared decks ≥30:** The relationship should have material support in the 584-deck corpus.
- **Jaccard ≥0.30:** The two cards should have meaningful overlap among decks containing either card.
- **Lift ≥1.20:** Co-occurrence must be at least 20% stronger than expected from independent popularity.
- **Shared archetypes ≥3:** The relationship should not be explained by one narrow deck strategy.
- **Mechanical/evolution pairs:** Declared before ranking and excluded from business recommendations.
- **Display Graph:** presentation only; centrality is computed on the full Analysis Graph.


In [ ]:
# 2) Load frozen snapshot and validate data quality
df = pd.read_csv(DATA_PATH)

N = df["deck_id"].nunique()
deck_totals = df.groupby("deck_id")["copies"].sum()

print("FROZEN DATASET")
print("Valid decklists:", N)
print("Card-line rows:", len(df))
print("Every deck totals 60 cards:", bool(deck_totals.eq(60).all()))

assert N == 584, f"Expected 584 decklists, found {N}"
assert len(df) == 14744, f"Expected 14,744 card-line rows, found {len(df)}"
assert deck_totals.eq(60).all(), "At least one deck does not total 60 cards."
assert "player" not in [c.lower() for c in df.columns], "Player identifier unexpectedly present."


In [ ]:
# 3) Technical cleaning: remove Basic Energy and build Deck × Card presence matrix
BASIC = {
    "grass energy", "fire energy", "water energy", "lightning energy",
    "psychic energy", "fighting energy", "darkness energy",
    "metal energy", "fairy energy"
}

def is_basic_energy(name):
    n = re.sub(r"\s+", " ", str(name).strip().lower())
    return (
        n in BASIC
        or (n.startswith("basic ") and n.endswith(" energy"))
        or n.startswith("basic {")
    )

nonbasic = df[~df["card"].map(is_basic_energy)].copy()

deck_card = nonbasic.groupby(["deck_id", "card"], as_index=False)["copies"].sum()
deck_card["present"] = 1

presence = (
    deck_card.pivot_table(
        index="deck_id",
        columns="card",
        values="present",
        aggfunc="max",
        fill_value=0
    ).astype(np.uint8)
)

print("Unique non-Basic-Energy cards:", presence.shape[1])
print("Deck × card matrix:", presence.shape)


In [ ]:
# 4) Graph A — raw / pre-pruning network
names_all = presence.columns.tolist()
X_all = presence.to_numpy(dtype=np.int32)
co_all = X_all.T @ X_all

raw_rows = []
for i, a in enumerate(names_all):
    for j in range(i + 1, len(names_all)):
        shared = int(co_all[i, j])
        if shared > 0:
            raw_rows.append((a, names_all[j], shared))

raw_edges = pd.DataFrame(raw_rows, columns=["card_a", "card_b", "shared_decks"])
Graw = nx.from_pandas_edgelist(raw_edges, "card_a", "card_b", edge_attr="shared_decks")

print("GRAPH A — PRE-PRUNING")
print("Nodes:", Graw.number_of_nodes())
print("Edges:", Graw.number_of_edges())

assert Graw.number_of_nodes() == 501
assert Graw.number_of_edges() == 18049


In [ ]:
# 5) Card prevalence + pair metrics (Jaccard, Lift, archetype breadth)
card_counts = presence.sum(axis=0).astype(int)
min_card_decks = math.ceil(MIN_CARD_PREVALENCE * N)

kept = card_counts[card_counts >= min_card_decks].index.tolist()
P = presence[kept].copy()

deck_arch = (
    df[["deck_id", "archetype"]]
    .drop_duplicates("deck_id")
    .set_index("deck_id")["archetype"]
)

names = P.columns.tolist()
X = P.to_numpy(dtype=np.int32)
co = X.T @ X

pair_rows = []
for i, a in enumerate(names):
    count_a = int(co[i, i])

    for j in range(i + 1, len(names)):
        b = names[j]
        shared = int(co[i, j])
        if shared == 0:
            continue

        count_b = int(co[j, j])
        union = count_a + count_b - shared
        jaccard = shared / union

        p_a = count_a / N
        p_b = count_b / N
        p_ab = shared / N
        lift = p_ab / (p_a * p_b)

        shared_ids = P.index[(P[a] == 1) & (P[b] == 1)]
        breadth = int(deck_arch.loc[shared_ids].nunique())

        pair_rows.append({
            "card_a": a,
            "card_b": b,
            "count_a": count_a,
            "count_b": count_b,
            "shared_decks": shared,
            "jaccard": jaccard,
            "lift": lift,
            "shared_archetypes": breadth,
            "mechanical_pair": tuple(sorted((a, b))) in MECHANICAL_PAIRS,
        })

pairs = pd.DataFrame(pair_rows)

print("Cards meeting prevalence threshold:", len(kept))
print("Minimum deck count:", min_card_decks)
assert len(kept) == 52
assert min_card_decks == 59


In [ ]:
# 6) Business-aware pruning → Analysis Graph
eligible = pairs[
    (pairs["shared_decks"] >= MIN_SHARED_DECKS)
    & (pairs["jaccard"] >= JACCARD_CUTOFF)
    & (pairs["lift"] >= MIN_LIFT)
    & (pairs["shared_archetypes"] >= MIN_SHARED_ARCHETYPES)
    & (~pairs["mechanical_pair"])
].copy().sort_values(
    ["jaccard", "shared_decks", "lift"],
    ascending=[False, False, False]
).reset_index(drop=True)

G = nx.Graph()
for r in eligible.itertuples(index=False):
    G.add_edge(
        r.card_a,
        r.card_b,
        jaccard=float(r.jaccard),
        shared_decks=int(r.shared_decks),
        lift=float(r.lift),
        shared_archetypes=int(r.shared_archetypes),
    )

print("ANALYSIS GRAPH")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Components:", nx.number_connected_components(G))

assert G.number_of_nodes() == 43
assert G.number_of_edges() == 115

eligible[
    ["card_a", "card_b", "shared_decks", "jaccard", "lift", "shared_archetypes"]
].head(10)


In [ ]:
# 7) Node metrics → Core Watchlist
bet = nx.betweenness_centrality(G, normalized=True, weight=None)

weighted_degree = {
    n: sum(d["jaccard"] for _, _, d in G.edges(n, data=True))
    for n in G.nodes()
}

prev = (P.sum(axis=0) / N).to_dict()
deck_count = P.sum(axis=0).astype(int).to_dict()

metrics = pd.DataFrame({
    "card": list(G.nodes()),
    "deck_count": [int(deck_count[n]) for n in G.nodes()],
    "prevalence": [float(prev[n]) for n in G.nodes()],
    "degree": [G.degree(n) for n in G.nodes()],
    "weighted_degree_jaccard": [weighted_degree[n] for n in G.nodes()],
    "betweenness": [bet[n] for n in G.nodes()],
}).sort_values(
    ["weighted_degree_jaccard", "prevalence", "betweenness"],
    ascending=False
).reset_index(drop=True)

metrics.head(10)


In [ ]:
# 8) Construction choice check: raw popularity vs business-aware relationship
raw_top = pairs.sort_values(
    ["shared_decks", "jaccard"],
    ascending=False
)[
    ["card_a", "card_b", "shared_decks", "jaccard", "lift", "shared_archetypes"]
].head(5)

print("TOP RAW CO-OCCURRENCE PAIRS")
display(raw_top)

print("\nTOP ELIGIBLE BUSINESS PAIRS")
display(
    eligible[
        ["card_a", "card_b", "shared_decks", "jaccard", "lift", "shared_archetypes"]
    ].head(5)
)


In [ ]:
# 9) Display backbone — visualization only
top_watch = metrics.head(TOP_WATCHLIST_NODES_FOR_DISPLAY)["card"].tolist()
top_pairs = eligible.head(TOP_PAIR_EDGES_FOR_DISPLAY)

display_nodes = set(top_watch)
for r in top_pairs.itertuples(index=False):
    display_nodes.add(r.card_a)
    display_nodes.add(r.card_b)

sub_edges = eligible[
    eligible["card_a"].isin(display_nodes)
    & eligible["card_b"].isin(display_nodes)
].copy()

display_keep = set()
for n in display_nodes:
    s = sub_edges[
        (sub_edges["card_a"] == n) | (sub_edges["card_b"] == n)
    ].sort_values(
        ["jaccard", "shared_decks", "lift"],
        ascending=[False, False, False]
    ).head(MAX_DISPLAY_EDGES_PER_NODE)

    for r in s.itertuples(index=False):
        display_keep.add(tuple(sorted((r.card_a, r.card_b))))

display_edges = sub_edges[
    sub_edges.apply(
        lambda r: tuple(sorted((r["card_a"], r["card_b"]))) in display_keep,
        axis=1
    )
].copy()

Gdisp = nx.Graph()
for r in display_edges.itertuples(index=False):
    Gdisp.add_edge(
        r.card_a,
        r.card_b,
        jaccard=float(r.jaccard),
        shared_decks=int(r.shared_decks),
        lift=float(r.lift),
        shared_archetypes=int(r.shared_archetypes),
    )

print("DISPLAY GRAPH")
print("Nodes:", Gdisp.number_of_nodes())
print("Edges:", Gdisp.number_of_edges())
print("Centrality above was computed on the 43-node Analysis Graph.")

assert Gdisp.number_of_nodes() == 12
assert Gdisp.number_of_edges() == 15


In [ ]:
# 10) Robustness check
best = eligible.iloc[0]
best_key = tuple(sorted((best["card_a"], best["card_b"])))

rob_rows = []
for cut in ROBUSTNESS_JACCARD:
    ed = pairs[
        (pairs["shared_decks"] >= MIN_SHARED_DECKS)
        & (pairs["jaccard"] >= cut)
        & (pairs["lift"] >= MIN_LIFT)
        & (pairs["shared_archetypes"] >= MIN_SHARED_ARCHETYPES)
        & (~pairs["mechanical_pair"])
    ].copy()

    gg = nx.Graph()
    for r in ed.itertuples(index=False):
        gg.add_edge(r.card_a, r.card_b)

    survives = any(
        tuple(sorted((r.card_a, r.card_b))) == best_key
        for r in ed.itertuples(index=False)
    )

    rob_rows.append({
        "jaccard_cutoff": cut,
        "nodes": gg.number_of_nodes(),
        "edges": gg.number_of_edges(),
        "components": nx.number_connected_components(gg) if gg.number_of_nodes() else 0,
        "top_pair_survives": survives,
    })

robustness = pd.DataFrame(rob_rows)
robustness


In [ ]:
# 11) Actionable graph for notebook inspection
plt.figure(figsize=(13, 8))
pos = nx.spring_layout(Gdisp, seed=42, weight="jaccard", k=1.4, iterations=150)

node_sizes = [900 + 3500 * float(prev.get(n, 0)) for n in Gdisp.nodes()]
edge_widths = [1.5 + 7 * Gdisp[u][v]["jaccard"] for u, v in Gdisp.edges()]

nx.draw_networkx_nodes(Gdisp, pos, node_size=node_sizes, alpha=0.92)
nx.draw_networkx_edges(Gdisp, pos, width=edge_widths, alpha=0.40)
nx.draw_networkx_labels(Gdisp, pos, font_size=9)

plt.title(
    "Actionable Stock Network — Pokémon TCG\n"
    f"Analysis: {G.number_of_nodes()} nodes / {G.number_of_edges()} edges | "
    f"Display: {Gdisp.number_of_nodes()} nodes / {Gdisp.number_of_edges()} edges"
)
plt.axis("off")
plt.tight_layout()

notebook_graph = OUT / "actionable_network_from_notebook.png"
plt.savefig(notebook_graph, dpi=220, bbox_inches="tight")
plt.show()

print("Saved:", notebook_graph)


In [ ]:
# 12) Submitted findings — machine-checked against the frozen snapshot
watch = metrics.iloc[0]

print("A4-READY FINDINGS")
print(
    f"Graph reduction: Raw {Graw.number_of_nodes()}/{Graw.number_of_edges()} -> "
    f"Analysis {G.number_of_nodes()}/{G.number_of_edges()} -> "
    f"Display {Gdisp.number_of_nodes()}/{Gdisp.number_of_edges()} (nodes/edges)"
)
print(
    f"Core Watchlist: {watch['card']} | {int(watch['deck_count'])}/{N} decks "
    f"({watch['prevalence']:.1%}) | "
    f"Weighted Degree={watch['weighted_degree_jaccard']:.3f} | "
    f"Betweenness={watch['betweenness']:.3f}"
)
print(
    f"Pair-check: {best['card_a']} — {best['card_b']} | "
    f"shared={int(best['shared_decks'])} | "
    f"Jaccard={best['jaccard']:.3f} | "
    f"Lift={best['lift']:.3f} | "
    f"breadth={int(best['shared_archetypes'])}"
)

assert watch["card"] == "Budew"
assert int(watch["deck_count"]) == 209
assert round(float(watch["prevalence"]), 3) == 0.358

assert {best["card_a"], best["card_b"]} == {"Dragapult ex", "Rosa's Encouragement"}
assert int(best["shared_decks"]) == 94
assert round(float(best["jaccard"]), 3) == 0.712
assert round(float(best["lift"]), 3) == 4.366
assert int(best["shared_archetypes"]) == 3

print("\nAll submitted headline numbers matched the frozen snapshot.")


## Business interpretation

- **Core Watchlist:** Budew — 209/584 decklists (35.8%), Weighted Degree 7.614, Betweenness 0.302
- **Pair-check:** Dragapult ex — Rosa's Encouragement — shared 94 decklists, Jaccard 0.712, Lift 4.366, breadth 3 archetypes
- **Robustness:** pair survives Jaccard cutoffs 0.25 / 0.30 / 0.35
- **Limitation:** competitive deck co-occurrence is not the same as customer co-purchase; validate with store POS before using the relationship for bundle/cross-sell decisions.
